**Assignment - Credit card defaults**

https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients

Import the dataset into a pandas dataframe. The first row consists of names that are
slightly more descriptive than just 'X1'. If you want, you can rename the column names
using the first row, then drop the first row. You can use df.rename() and
df.drop(df.index[0], axis=0)  
The dataset contains a column named Unnamed: 0, which is just an index. You can throw
this column away using df.drop.

The code given on the webpage is loading the data from the web.
  
Below is code to read data from xls (extracted from zip from web)

In [78]:
import pandas as pd

df = pd.read_excel('data/default of credit card clients.xls', header=1)
# PAY_x is not consistent with PAY_AMTx
# target has long name
df.rename(columns={'PAY_0': 'PAY_1',
                   'default payment next month': 'IS_DEFAULT'
                   }, inplace=True)
# drop the first column (ID)
df.drop(columns=['ID'], inplace=True)

# split it in X and y
X = df.drop(columns=['IS_DEFAULT'])
y = df['IS_DEFAULT']

Split the dataset into a train-, and test set, using  
`train_test_split(X, y, test_size=0.5, random_state=42)`

In [79]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(15000, 23) (15000,) (15000, 23) (15000,)


 Write a preprocessor that  
• applies a one-hot encoding to process the categorical variables.  
• rescales the features that are not categorical using a standard scale

First, check which columns have limited unique values, these are likely
to be categorical

In [80]:
print(df.nunique())

categorical_cols = [col for col in X.columns if X[col].nunique() < 10]
print(categorical_cols)

LIMIT_BAL        81
SEX               2
EDUCATION         7
MARRIAGE          4
AGE              56
PAY_1            11
PAY_2            11
PAY_3            11
PAY_4            11
PAY_5            10
PAY_6            10
BILL_AMT1     22723
BILL_AMT2     22346
BILL_AMT3     22026
BILL_AMT4     21548
BILL_AMT5     21010
BILL_AMT6     20604
PAY_AMT1       7943
PAY_AMT2       7899
PAY_AMT3       7518
PAY_AMT4       6937
PAY_AMT5       6897
PAY_AMT6       6939
IS_DEFAULT        2
dtype: int64
['SEX', 'EDUCATION', 'MARRIAGE']


So, from SEX, EDUCATION and MARRIAGE the one-hot encoding can be done, all other standard scaler

In [82]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

categorical_cols = [col for col in X.columns if X[col].nunique() < 10]
other_cols = [col for col in X.columns if X[col].nunique() >= 10]

cat_pipeline = make_pipeline(
  OneHotEncoder(handle_unknown='ignore'),
)
standard_pipeline = make_pipeline(
  StandardScaler(),
)

preprocessor = ColumnTransformer([
  ("cat", cat_pipeline, categorical_cols),
  ("", standard_pipeline, other_cols)
])

X_prep = preprocessor.fit_transform(X_train)

print(preprocessor.get_feature_names_out())
print(X_prep.shape)


['cat__SEX_1' 'cat__SEX_2' 'cat__EDUCATION_0' 'cat__EDUCATION_1'
 'cat__EDUCATION_2' 'cat__EDUCATION_3' 'cat__EDUCATION_4'
 'cat__EDUCATION_5' 'cat__EDUCATION_6' 'cat__MARRIAGE_0' 'cat__MARRIAGE_1'
 'cat__MARRIAGE_2' 'cat__MARRIAGE_3' '__LIMIT_BAL' '__AGE' '__PAY_1'
 '__PAY_2' '__PAY_3' '__PAY_4' '__PAY_5' '__PAY_6' '__BILL_AMT1'
 '__BILL_AMT2' '__BILL_AMT3' '__BILL_AMT4' '__BILL_AMT5' '__BILL_AMT6'
 '__PAY_AMT1' '__PAY_AMT2' '__PAY_AMT3' '__PAY_AMT4' '__PAY_AMT5'
 '__PAY_AMT6']
(15000, 33)
